# Deep Learning Models — ANN, Autoencoder, LSTM

Tests `src/models/deep_learning.py` against the real feature table.

In [ ]:
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("Working directory:", os.getcwd())

import sys
sys.path.insert(0, '.')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader

from src.utils.config import load_config
from src.data.load import load_raw_transactions
from src.data.clean import run_cleaning_pipeline
from src.features.build_features import build_customer_feature_table
from src.features.preprocessing import stratified_split, get_feature_columns, fit_scaler, apply_scaler
from src.models.deep_learning import (
    train_ann, predict_ann_proba, train_autoencoder,
    compute_reconstruction_error, train_lstm, PurchaseSequenceDataset,
)

## 1. Rebuild the feature table and scaled splits

In [ ]:
cfg = load_config('config/config.yaml')
raw = load_raw_transactions(cfg['paths']['raw_data'], cfg['sheets'])
cleaned = run_cleaning_pipeline(raw, cfg['cleaning']['non_product_stockcodes'], cfg['cleaning']['outlier_iqr_multiplier'])

cutoff = cleaned['invoice_date'].max() - pd.Timedelta(days=90)
feats = build_customer_feature_table(cleaned, cutoff, churn_window_days=90)

train, val, test = stratified_split(feats)
feature_cols = [c for c in get_feature_columns(feats, exclude=['customer_id', 'churned'])
                 if not c.startswith('affinity_')]

scaler = fit_scaler(train)
X_train = apply_scaler(train[feature_cols], scaler).values.astype('float32')
X_val = apply_scaler(val[feature_cols], scaler).values.astype('float32')
X_test = apply_scaler(test[feature_cols], scaler).values.astype('float32')
y_train = train['churned'].values.astype('float32')
y_val = val['churned'].values.astype('float32')
y_test = test['churned'].values.astype('float32')

print('X_train:', X_train.shape, '| positive rate:', y_train.mean().round(3))

## 2. Train the ANN churn classifier

In [ ]:
ann_model, ann_history = train_ann(X_train, y_train, X_val, y_val, max_epochs=200, patience=15)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ann_history.train_loss, label='train loss')
ax.plot(ann_history.val_loss, label='val loss')
ax.set_xlabel('epoch')
ax.set_ylabel('loss')
ax.set_title('ANN training curve (early stopping)')
ax.legend()
plt.show()

print('Trained for', len(ann_history.train_loss), 'epochs before early stopping')

## 3. Evaluate the ANN on the held-out test set

In [ ]:
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score

test_proba = predict_ann_proba(ann_model, X_test)
test_pred = (test_proba >= 0.5).astype(int)

print('ROC-AUC:', roc_auc_score(y_test, test_proba).round(4))
print('Precision:', precision_score(y_test, test_pred).round(4))
print('Recall:', recall_score(y_test, test_pred).round(4))
print('F1:', f1_score(y_test, test_pred).round(4))

## 4. Autoencoder — anomaly detection on spending patterns

In [ ]:
ae_model, ae_history, anomaly_threshold = train_autoencoder(X_train, X_val, max_epochs=150, patience=15)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ae_history.train_loss, label='train loss')
ax.plot(ae_history.val_loss, label='val loss')
ax.set_title('Autoencoder training curve')
ax.legend()
plt.show()

test_errors = compute_reconstruction_error(ae_model, X_test)
n_anomalies = (test_errors > anomaly_threshold).sum()
print(f'Anomaly threshold (95th pct of val error): {anomaly_threshold:.4f}')
print(f'Flagged {n_anomalies} / {len(test_errors)} test customers as anomalous ({n_anomalies/len(test_errors):.1%})')

## 5. LSTM — purchase-timing sequence model (log-space target)

**Update**: the raw day-gap target is heavy-tailed (0 to 700+ days), which
dominated the MSE loss and gave a poor RMSE of ~99 days against a median
gap of only 16 days. We now train on `log1p(gap_days)` instead, which
compresses the tail, then convert predictions back to real days
(`expm1`) for honest evaluation.

In [ ]:
def build_purchase_sequences(cleaned_df, customer_ids, cutoff_date, max_products=10):
    valid = cleaned_df[~cleaned_df['is_missing_customer_id'] & ~cleaned_df['is_non_product']
                        & (cleaned_df['invoice_date'] < cutoff_date)].copy()
    valid['line_total'] = valid['quantity'] * valid['price']

    top_codes = (valid.groupby('stock_code')['line_total'].sum()
                       .nlargest(max_products).index.tolist())
    code_to_idx = {c: i for i, c in enumerate(top_codes)}
    valid['category_code'] = valid['stock_code'].map(code_to_idx).fillna(max_products).astype(float)

    orders = (valid.groupby(['customer_id', 'invoice'])
                    .agg(invoice_date=('invoice_date', 'min'), amount=('line_total', 'sum'),
                         category_code=('category_code', 'first'))
                    .reset_index()
                    .sort_values(['customer_id', 'invoice_date']))
    orders['gap_days'] = orders.groupby('customer_id')['invoice_date'].diff().dt.days.fillna(0)

    sequences, targets, seq_customer_ids = [], [], []
    for cid, group in orders.groupby('customer_id'):
        if cid not in customer_ids or len(group) < 2:
            continue
        seq = group[['amount', 'gap_days', 'category_code']].values[:-1].astype('float32')
        next_gap = group['gap_days'].values[-1]
        if len(seq) == 0 or next_gap <= 0:
            continue
        sequences.append(seq)
        targets.append(next_gap)
        seq_customer_ids.append(cid)
    return sequences, np.array(targets, dtype='float32'), seq_customer_ids


train_seqs, train_targets, _ = build_purchase_sequences(cleaned, set(train['customer_id']), cutoff)
val_seqs, val_targets, _ = build_purchase_sequences(cleaned, set(val['customer_id']), cutoff)
print(f'Built {len(train_seqs)} training sequences, {len(val_seqs)} validation sequences')
print(f'Median actual gap: {np.median(train_targets):.1f} days | max: {train_targets.max():.0f} days')

In [ ]:
# Train in log-space to tame the heavy tail
train_targets_log = np.log1p(train_targets)
val_targets_log = np.log1p(val_targets)

lstm_model, lstm_history = train_lstm(train_seqs, train_targets_log, val_seqs, val_targets_log,
                                       max_epochs=100, patience=10)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(lstm_history.train_loss, label='train loss (MSE, log-days^2)')
ax.plot(lstm_history.val_loss, label='val loss')
ax.set_title('LSTM training curve (log-space target)')
ax.legend()
plt.show()

### Evaluate in real days (inverse-transformed)

This is the number that actually matters — RMSE/MAE converted back to
days, comparable against the median 16-day gap we saw in EDA.

In [ ]:
def predict_lstm(model, sequences, max_len):
    device = next(model.parameters()).device
    dummy_targets = np.zeros(len(sequences), dtype='float32')
    ds = PurchaseSequenceDataset(sequences, dummy_targets, max_len=max_len)
    loader = DataLoader(ds, batch_size=64, shuffle=False)
    model.eval()
    preds = []
    with torch.no_grad():
        for xb, lengths, _ in loader:
            xb = xb.to(device)
            out = model(xb, lengths)
            preds.append(out.cpu().numpy())
    return np.concatenate(preds)


max_len = max(len(s) for s in train_seqs)
val_preds_log = predict_lstm(lstm_model, val_seqs, max_len=max_len)
val_preds_days = np.clip(np.expm1(val_preds_log), 0, None)  # back to real days

from sklearn.metrics import mean_absolute_error, mean_squared_error

rmse_days = mean_squared_error(val_targets, val_preds_days) ** 0.5
mae_days = mean_absolute_error(val_targets, val_preds_days)

print(f'RMSE (real days): {rmse_days:.1f}')
print(f'MAE (real days): {mae_days:.1f}')
print(f'Median actual gap: {np.median(val_targets):.1f} days')
print(f'Naive baseline (always predict median): '
      f'RMSE={mean_squared_error(val_targets, np.full_like(val_targets, np.median(train_targets)))**0.5:.1f}, '
      f'MAE={mean_absolute_error(val_targets, np.full_like(val_targets, np.median(train_targets))):.1f}')